<a href="https://colab.research.google.com/github/kanavG10/adsrp-parkinson-s/blob/main/02_keypoint_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gait keypoint extraction (MediaPipe BlazePose)

### What this whole notebook does

It watches a video of somebody walking and figures out **where their body parts are
in every single frame** — left knee here, right wrist there, 33 points in total.
Then it draws a stick figure on the video so you can see it worked, and makes some
charts from the numbers.

Think of it as turning a video into a spreadsheet of body positions.

**Why we want that:** the V-JEPA masking study needs to hide specific body parts
during training (legs only, arms only, etc.). You can't hide a knee until you know
which pixels *are* the knee. This notebook is that first step.

**Run the cells top to bottom.** Takes about 5 minutes. No GPU needed — leave the
runtime on CPU, a GPU won't make it faster.

---


## Step 1 — Get the code and install the tools

**In plain English:** Colab hands you a brand-new empty computer every time you open
it. Nothing is saved between sessions. So the first cell copies our code down from
GitHub and installs the libraries it needs.

You have to run this every single time you open the notebook. It's not optional
setup you can skip once it's "done".


In [ ]:
!git clone -q https://github.com/kanavG10/adsrp-parkinson-s.git repo
%cd repo
!pip install -q -r requirements.txt
print('installed')


> **If you see a scary red box about conflicts and a `Restart session` button:**
> click it, then re-run starting from the `%cd repo` cell above. This is normal.
> MediaPipe wants slightly different versions of some libraries than Colab ships
> with, and restarting sorts it out. Your cloned files survive the restart.


## Step 2 — Check what actually installed

**In plain English:** a receipt. It prints the version numbers so that if something
breaks later, you can tell whether you got the versions we expect.

That last line checks for a specific trap. The old, popular way of using MediaPipe
was `mp.solutions.pose` — it's in basically every tutorial and YouTube video online.
**It was deleted from the library.** So it should print `False`, and any tutorial code
you copy from the internet will crash. Our code uses the newer replacement.


In [ ]:
import mediapipe as mp, cv2, sys
print('python    ', sys.version.split()[0])
print('mediapipe ', mp.__version__)
print('opencv    ', cv2.__version__)
print('mp.solutions present:', hasattr(mp, 'solutions'))   # expect False


## Step 3 — Download the actual AI model

**In plain English:** Step 1 installed the *software that can run* a pose model.
This downloads the pose model itself — a 9 MB file containing the trained weights
that actually know what a human elbow looks like.

It's like the difference between installing a DVD player and owning a DVD. Step 1
was the player; this is the disc.

There are three sizes — `lite`, `full`, `heavy`. We use `full`, the middle one.
`heavy` is a bit more accurate on far-away people but noticeably slower.


In [ ]:
!mkdir -p models
!wget -q -O models/pose_landmarker_full.task \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
!ls -lh models/


## Step 4 — Get a walking video

**In plain English:** downloads one video of a person walking down a hall.

There's a small trick here. The dataset is published as **one giant 2.35 GB zip file**
containing 28 videos. We only want one of them. Rather than download all 2.35 GB and
throw away 95% of it, `remotezip` reaches *inside* the zip over the internet and pulls
out just the one file we asked for (~100 MB). Like taking one book off a shelf instead
of buying the whole library.

⚠️ **These are not Parkinson's videos.** They're ordinary adults walking, with no
severity scores attached. They prove the pipeline works — they can't train or test
the real model. Getting properly labelled clinical video is still the blocker.

To use your own video instead: upload it into the `data/raw/` folder (file browser on
the left), then change the filenames in the cells below.


In [ ]:
import os
from remotezip import RemoteZip

URL  = 'https://ndownloader.figshare.com/files/35454242'
WANT = 'Videos/OAW01-bottom.mp4'
os.makedirs('data/raw', exist_ok=True)
out = 'data/raw/' + os.path.basename(WANT)

if not os.path.exists(out):
    with RemoteZip(URL) as z:
        with z.open(WANT) as src, open(out, 'wb') as dst:
            while chunk := src.read(1 << 20):
                dst.write(chunk)
print(out, round(os.path.getsize(out)/1e6, 1), 'MB')


## Step 5 — The main event: find the body points

**In plain English:** this goes through the video frame by frame and writes down the
position of 33 body points in each one. It produces two things:

1. a **table of numbers** (`outputs/keypoints/...parquet`) — the real output
2. a **video with a stick figure drawn on it** (`outputs/annotated/...mp4`) — so you
   can eyeball whether it worked

`--seconds 25` means only do the first 25 seconds, to keep this demo quick. Delete
that flag to do the whole 75-second video (a few minutes on Colab).

**Why this was harder than it sounds.** The person is walking far down a long hall,
so she's only about 100 pixels tall in a huge 1080×1920 frame — a tiny speck.
MediaPipe simply fails to find people that small; on raw frames it only worked
**47% of the time.**

The fix: the camera never moves, so we build a picture of the empty room (by taking
the median of frames sampled across the video — the person moves, the room doesn't,
so she averages out). Anything that differs from the empty room is the person. We crop
tightly to her, blow that crop up, *then* look for the pose. That gets it to ~100%.

Watch the `detected=` counter climb as it runs — it should match the frame number.


In [ ]:
!python src/extract_pose.py data/raw/OAW01-bottom.mp4 --model full --seconds 25


## Step 6 — Make a zoomed-in version

**In plain English:** the video from Step 5 shows the whole hall, so the stick figure
is a tiny scribble in the middle. This makes a second video that crops in and *follows*
her as she walks, so you can actually see the skeleton.

This is fast (seconds, not minutes) because it doesn't re-run the AI. It just reuses
the numbers already saved in Step 5 and redraws them zoomed in.

**This is the version to show people.**


In [ ]:
!python src/render_followcam.py data/raw/OAW01-bottom.mp4


## Step 7 — Watch the videos right here in the page

**In plain English:** a helper that plays a video inside the notebook.

It exists because of an annoying incompatibility: the tool that *writes* our video
saves it in a format Colab's built-in player **cannot play** — you'd get a black box.
So this converts a 10-second chunk into H.264 (the format browsers understand) and
embeds that.

It only shows a short excerpt on purpose. Embedding a whole 30 MB video would make
this notebook enormous and slow to open.


In [ ]:
import subprocess, base64
from IPython.display import HTML, display

def show(path, start=8, dur=10, width=320):
    """Convert a clip to a browser-friendly format and play it inline."""
    out = '/tmp/preview.mp4'
    subprocess.run(['ffmpeg','-y','-loglevel','error','-ss',str(start),
                    '-i',path,'-t',str(dur),'-vcodec','libx264',
                    '-pix_fmt','yuv420p','-crf','24',out], check=True)
    b64 = base64.b64encode(open(out,'rb').read()).decode()
    display(HTML(f'<video width={width} controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

show('outputs/annotated/OAW01-bottom_followcam.mp4')


**Colours are not decoration** — they're the masking groups from the study:

| colour | body part | used by |
|---|---|---|
| 🟠 orange | arms, hands | Model 3 (arm swing) |
| 🟢 green | legs, feet | Model 2 (lower body) |
| 🔵 blue | torso | — |
| ⚪ grey | face, head | Model 6 (control) |

So the video itself previews what each experiment would hide.


### The same thing un-zoomed

**In plain English:** this is the honest version — the real frame, uncropped. It shows
how small she actually is, which is why all that cropping machinery was necessary.


In [ ]:
show('outputs/annotated/OAW01-bottom_pose.mp4', width=200)


## Step 8 — Turn the numbers into charts

**In plain English:** reads the table of body positions and draws three pictures.
It also prints a one-line summary, including **cadence** (steps per minute).

Cadence is a good reality check. A normal adult walks at roughly 100–120 steps per
minute. If this printed 12 or 900, something would be badly wrong. It prints two
different estimates and **warns you if they disagree** — better to admit uncertainty
than to print a confident wrong number.


In [ ]:
!python src/gait_report.py outputs/keypoints/OAW01-bottom.parquet


**In plain English:** displays the three charts that the cell above just saved.

1. **quality** — a heat map of how confident the model was about each body part over
   time. Yellow = confident, dark = struggling. You'll see dark stripes on the hands:
   they vanish when they swing behind her body. **This matters for the study** — the
   arm points Model 3 wants to mask are the least reliable ones we have.
2. **gait** — the actual walking signal. The top wiggly lines are her ankles going up
   and down; each bump is a step. The bottom chart finds the rhythm of those bumps.
3. **masking_conditions** — all six experiments drawn on her real skeleton. Red circles
   = joints that would be hidden during training.


In [ ]:
from IPython.display import Image, display
for name in ['quality','gait','masking_conditions']:
    display(Image(f'outputs/figures/OAW01-bottom_{name}.png', width=1000))


## Step 9 — Look at the raw numbers

**In plain English:** a peek at the actual spreadsheet. Every row is one body point in
one frame, so 25 seconds of video becomes ~24,000 rows.

The columns worth knowing:
- `x`, `y` — where the point is *in the picture* (0 to 1, like a percentage across)
- `visibility` — how sure the model is (1.0 = certain, 0.2 = guessing)
- `wx`, `wy`, `wz` — position in **real-world 3D metres**, measured from the hips

**Use `wx/wy/wz` for any real measurement, not `x/y`.** Because she walks toward and
away from the camera, she gets bigger and smaller in the picture — so `x/y` distances
lie to you. The world coordinates already account for that.


In [ ]:
import pandas as pd
df = pd.read_parquet('outputs/keypoints/OAW01-bottom.parquet')
print(df.shape, '=', df.frame.nunique(), 'frames x', df.joint_id.nunique(), 'joints')
df.head()


## Step 10 — The six masking recipes

**In plain English:** prints the six experiments, listing which body parts each one
**hides during training**. All 33 points are visible again at test time — the hiding
only happens while the model is learning.

The idea being tested: if you force the model to guess the legs from everything else,
does it learn more about walking than if you'd hidden random patches? Each recipe maps
to what doctors actually score on MDS-UPDRS Item 3.10 (stride length, stride speed,
foot lift, heel strike, turning, arm swing).

`face_control` is the deliberate dud — the face has nothing to do with walking, so if
hiding it works *just as well*, the whole hypothesis is in trouble.


In [ ]:
sys.path.insert(0, 'src')
from pose_topology import MASK_GROUPS, LANDMARK_NAMES
for name, idxs in MASK_GROUPS.items():
    joints = ', '.join(LANDMARK_NAMES[i] for i in idxs[:5])
    print(f'{name:14s} masks {len(idxs):2d}/33  ({joints}, ...)')


---
## Things to know

**Colab forgets everything when the session ends.** Your `outputs/` folder disappears.
To keep results, either mount Google Drive and copy them there:
```python
from google.colab import drive; drive.mount('/content/drive')
!cp -r outputs /content/drive/MyDrive/gait_outputs
```
or just re-run the notebook when you need them again.

**Using your own videos.** Upload to `data/raw/` and change the filenames. One catch:
the trick from Step 5 assumes the **camera never moves**. For handheld or phone video
add `--no-roi` to the Step 5 command, and expect it to find fewer poses.

**Known limitations, worth being upfront about:**
- Hands and fingers track worst (~0.67–0.73 confidence) and get worse the further away
  the person is. Those are exactly the points Model 3 masks.
- Faces in this dataset are **blurred for privacy**, so the face points drift around.
  If the real clinical videos are blurred too, `face_control` would be masking points
  that were never reliable — which would make it a broken control rather than a fair
  comparison. Worth checking before depending on it.
- Model 2 was written up as "12 joints" but the listed points 23–32 are only **10**.
  MediaPipe has no toe landmarks; heel + foot index cover foot lift and heel strike.
